# LeetCode 609 — Find Duplicate File in System bằng Python

Link đề: https://leetcode.com/problems/find-duplicate-file-in-system/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String

Skeleton LeetCode:

```python
class Solution:
    def findDuplicate(self, paths: List[str]) -> List[List[str]]:
        ...
```

Ghi chú: bài có **nhiều đáp án đúng** (đề cho phép trả các nhóm theo **thứ tự bất kỳ**, và thứ tự đường dẫn trong mỗi nhóm cũng không quan trọng). Vì vậy notebook test bằng **hàm chuẩn hoá** (sort từng nhóm, rồi sort các nhóm) và **hàm kiểm tra tính hợp lệ**, không so cứng với output mẫu.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: mỗi chuỗi trong `paths` mô tả một thư mục và các file kèm nội dung `tên(nội_dung)`; cần gom các file **cùng nội dung** thành nhóm (nhóm phải có ≥ 2 file) và trả về **đường dẫn đầy đủ** của chúng.
2. Nắm các công cụ Python: `str.split()`, `str.find`, `str.partition`, cắt chuỗi (slicing, kể cả chỉ số âm `[:-1]`), `defaultdict(list)`, `sorted` + `itertools.groupby`, `hashlib`, và cách **chuẩn hoá** đáp án để test bài nhiều đáp án đúng.
3. Đi từ **so sánh từng cặp file** → **bảng băm nội dung → danh sách đường dẫn** → **sắp xếp theo nội dung rồi gom nhóm liên tiếp** → **lọc nhiều tầng (kích thước → đầu file → băm → so từng byte)** cho file khổng lồ.
4. Hiểu mấu chốt: bài toán là **"nhóm theo khoá" (group by)** — chọn đúng khoá là *nội dung file*, rồi bỏ nhóm chỉ có 1 phần tử.
5. Tránh các lỗi kinh điển: quên lọc nhóm kích thước 1, quên ghép thư mục vào tên file, dùng chung một list cho mọi khoá (`dict.fromkeys(keys, [])`), và test bài nhiều đáp án bằng so sánh cứng theo thứ tự.

## 2. Đề bài & ví dụ

Cho danh sách `paths`. Mỗi phần tử có dạng:

```
"root/d1/d2/.../dm f1.txt(f1_content) f2.txt(f2_content) ... fn.txt(fn_content)"
```

nghĩa là thư mục `root/d1/d2/.../dm` chứa `n` file `f1.txt, ..., fn.txt` với nội dung tương ứng (`n >= 1`, `m >= 0`; nếu `m = 0` thì thư mục chính là `root`). Giữa đường dẫn thư mục và các file, và giữa các file, ngăn cách bằng **một dấu cách**.

Hãy trả về **mọi nhóm file trùng nội dung**, mỗi file được biểu diễn bằng `"đường_dẫn_thư_mục/tên_file"`. Một nhóm có **ít nhất hai** file cùng nội dung. Có thể trả về theo **thứ tự bất kỳ**.

**Ví dụ 1:**

```
paths = ["root/a 1.txt(abcd) 2.txt(efgh)","root/c 3.txt(abcd)","root/c/d 4.txt(efgh)","root 4.txt(efgh)"]
Output: [["root/a/2.txt","root/c/d/4.txt","root/4.txt"],["root/a/1.txt","root/c/3.txt"]]
```

**Ví dụ 2:**

```
paths = ["root/a 1.txt(abcd) 2.txt(efgh)","root/c 3.txt(abcd)","root/c/d 4.txt(efgh)"]
Output: [["root/a/2.txt","root/c/d/4.txt"],["root/a/1.txt","root/c/3.txt"]]
```

**Ràng buộc:**

- `1 <= paths.length <= 2 * 10^4`
- `1 <= paths[i].length <= 3000`, `1 <= sum(paths[i].length) <= 5 * 10^5`
- `paths[i]` gồm chữ cái, chữ số, `'/'`, `'.'`, `'('`, `')'` và dấu cách.
- Trong cùng một thư mục không có hai file/thư mục trùng tên; mỗi chuỗi mô tả một thư mục duy nhất.

Cell dưới chạy thử dữ liệu của hai ví dụ (chưa giải) và dựng hai hàm hỗ trợ dùng cho cả notebook: hàm **chuẩn hoá** đáp án và hàm **kiểm tra tính hợp lệ**.

In [1]:
import hashlib
import random
import re
import time
from collections import defaultdict
from itertools import groupby
from typing import List

ex1 = ["root/a 1.txt(abcd) 2.txt(efgh)", "root/c 3.txt(abcd)", "root/c/d 4.txt(efgh)", "root 4.txt(efgh)"]
ex2 = ["root/a 1.txt(abcd) 2.txt(efgh)", "root/c 3.txt(abcd)", "root/c/d 4.txt(efgh)"]
out1 = [["root/a/2.txt", "root/c/d/4.txt", "root/4.txt"], ["root/a/1.txt", "root/c/3.txt"]]
out2 = [["root/a/2.txt", "root/c/d/4.txt"], ["root/a/1.txt", "root/c/3.txt"]]


def norm(groups):
    """Chuẩn hoá đáp án: sort trong từng nhóm, rồi sort các nhóm -> so sánh được dù thứ tự khác nhau."""
    return sorted(sorted(g) for g in groups)


# Output mẫu của đề và một đáp án khác thứ tự: sau chuẩn hoá phải bằng nhau
shuffled = [["root/c/3.txt", "root/a/1.txt"], ["root/4.txt", "root/c/d/4.txt", "root/a/2.txt"]]
print(norm(out1))
print(norm(out1) == norm(shuffled), "(cùng nội dung, khác thứ tự)")
print(len(ex1), "chuỗi thư mục trong ví dụ 1;", sum(len(p.split()) - 1 for p in ex1), "file")

[['root/4.txt', 'root/a/2.txt', 'root/c/d/4.txt'], ['root/a/1.txt', 'root/c/3.txt']]
True (cùng nội dung, khác thứ tự)
4 chuỗi thư mục trong ví dụ 1; 5 file


## 3. Kiến thức nền

### 3.1. `str.split()` — tách chuỗi theo dấu cách

`s.split()` (không đối số) tách theo **mọi khoảng trắng** và bỏ phần rỗng; `s.split(" ")` tách đúng theo **một dấu cách**. Phần tử đầu là đường dẫn thư mục, các phần tử sau là `tên(nội_dung)`.

In [2]:
line = "root/a 1.txt(abcd) 2.txt(efgh)"
parts = line.split()
print(parts)
print("thư mục:", parts[0], "| các file:", parts[1:])

['root/a', '1.txt(abcd)', '2.txt(efgh)']
thư mục: root/a | các file: ['1.txt(abcd)', '2.txt(efgh)']


### 3.2. Tách `tên(nội_dung)` bằng `find` + slicing (và `partition`)

- `s.find("(")` trả **chỉ số** của ký tự `(` đầu tiên (hoặc `-1` nếu không có).
- Cắt chuỗi: `s[a:b]` lấy từ chỉ số `a` đến `b-1`; `s[:i]` là phần đứng **trước** vị trí `i`; chỉ số **âm** đếm từ cuối: `s[-1]` là ký tự cuối, `s[i+1:-1]` bỏ ký tự đầu (`(`) và ký tự cuối (`)`).
- `s.partition("(")` tách thành 3 phần `(trước, "(", sau)` — gọn hơn.

In [3]:
token = "1.txt(abcd)"
i = token.find("(")
name, content = token[:i], token[i + 1:-1]
print(i, repr(name), repr(content))            # 5 '1.txt' 'abcd'

name2, _, rest = token.partition("(")
print(repr(name2), repr(rest[:-1]))            # cùng kết quả bằng partition
print(token[-1], token[:-1])                   # ')'  và chuỗi bỏ ký tự cuối

5 '1.txt' 'abcd'
'1.txt' 'abcd'
) 1.txt(abcd


### 3.3. Ghép đường dẫn đầy đủ

Đường dẫn file = `thư_mục + "/" + tên_file`. Với `root 4.txt(efgh)` (thư mục chính là `root`, `m = 0`) ta được `root/4.txt`.

In [4]:
directory, name = "root/c/d", "4.txt"
print(directory + "/" + name)                  # root/c/d/4.txt
print("root" + "/" + "4.txt")                  # root/4.txt

root/c/d/4.txt
root/4.txt


### 3.4. `defaultdict(list)` — "nhóm theo khoá"

`d = defaultdict(list)`: khi gọi `d[k]` với khoá `k` chưa có, Python **tự tạo** `d[k] = []`. Nhờ đó `d[key].append(x)` gom các `x` cùng `key` mà không cần `if key not in d`. Đây là mẫu **group by** kinh điển.

In [5]:
d = defaultdict(list)
for word in ["apple", "avocado", "banana", "blueberry", "cherry"]:
    d[word[0]].append(word)                    # khoá = chữ cái đầu
print(dict(d))
print([v for v in d.values() if len(v) > 1])   # chỉ giữ nhóm có >= 2 phần tử

{'a': ['apple', 'avocado'], 'b': ['banana', 'blueberry'], 'c': ['cherry']}
[['apple', 'avocado'], ['banana', 'blueberry']]


### 3.5. `sorted` + `itertools.groupby` — gom nhóm bằng sắp xếp

`groupby(iterable, key)` gom các phần tử **liền nhau** có cùng khoá. Vì chỉ gom phần tử **liền kề**, ta phải `sorted` theo cùng khoá trước. Mỗi lần lặp trả `(khoá, iterator_các_phần_tử)`.

In [6]:
pairs = [("b", 1), ("a", 2), ("b", 3), ("a", 4)]
print([(k, list(g)) for k, g in groupby(pairs, key=lambda p: p[0])])                   # KHÔNG sort: 4 nhóm lẻ tẻ
pairs.sort(key=lambda p: p[0])
print([(k, [x for _, x in g]) for k, g in groupby(pairs, key=lambda p: p[0])])         # đã sort: 2 nhóm

[('b', [('b', 1)]), ('a', [('a', 2)]), ('b', [('b', 3)]), ('a', [('a', 4)])]
[('a', [2, 4]), ('b', [1, 3])]


### 3.6. Hàm băm `hashlib` (cho phần follow-up về file lớn)

`hashlib.sha256(bytes).hexdigest()` biến dữ liệu bất kỳ thành chuỗi 64 ký tự "dấu vân tay". Hai nội dung **giống nhau** luôn cho cùng dấu vân tay; hai nội dung **khác nhau** gần như chắc chắn khác (nhưng về lý thuyết có thể đụng — nên cẩn thận phải kiểm tra lại, mục 7). Có thể băm **từng đoạn nhỏ** bằng `update()` để không phải nạp cả file vào RAM.

In [7]:
whole = hashlib.sha256(b"abcdefgh").hexdigest()
h = hashlib.sha256()
for chunk in (b"abcd", b"efgh"):               # đọc từng đoạn 4 byte
    h.update(chunk)
print(h.hexdigest() == whole, whole[:16], "...")
print(hashlib.sha256(b"abcdefgx").hexdigest()[:16], "... (khác)")

True 9c56cc51b374c3ba ...
fa7497688e7f800b ... (khác)


### 3.7. Test bài nhiều đáp án đúng: chuẩn hoá hoặc kiểm tra tính hợp lệ

Đáp án của ta có thể khác output mẫu về **thứ tự**. Có hai cách test đúng:

1. **Chuẩn hoá** (`norm` ở mục 2): sort trong nhóm rồi sort các nhóm, so sánh hai đáp án.
2. **Kiểm tra tính hợp lệ** bằng một hàm riêng không phụ thuộc cách giải (mục 12): mỗi nhóm ≥ 2 file, cùng nội dung, bao phủ mọi file trùng, không có file xuất hiện hai lần.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy làm tay với **ví dụ 1**. Đầu tiên "dịch" mọi chuỗi thành danh sách (đường dẫn đầy đủ, nội dung):

| thư mục | file | đường dẫn đầy đủ | nội dung |
|---|---|---|---|
| `root/a` | `1.txt` | `root/a/1.txt` | `abcd` |
| `root/a` | `2.txt` | `root/a/2.txt` | `efgh` |
| `root/c` | `3.txt` | `root/c/3.txt` | `abcd` |
| `root/c/d` | `4.txt` | `root/c/d/4.txt` | `efgh` |
| `root` | `4.txt` | `root/4.txt` | `efgh` |

Rồi "xếp vào hộp theo nội dung":

- hộp `abcd`: `root/a/1.txt`, `root/c/3.txt`
- hộp `efgh`: `root/a/2.txt`, `root/c/d/4.txt`, `root/4.txt`

Hộp nào có **từ 2 file trở lên** thì là nhóm trùng ⇒ cả hai hộp đều là đáp án. Chính là ví dụ 1!

Với **ví dụ 2** (thiếu `root/4.txt`): hộp `efgh` còn 2 file, hộp `abcd` còn 2 file ⇒ vẫn 2 nhóm.

Một ví dụ không có trùng: `["root a.txt(x) b.txt(y)"]` ⇒ hộp `x` có 1 file, hộp `y` có 1 file ⇒ đáp án `[]`.

Điểm cốt lõi: **khoá để gom nhóm là *nội dung***, không phải tên file hay thư mục. Phần còn lại chỉ là xử lý chuỗi cẩn thận.

## 5. Cách 1 — Brute Force / baseline: so sánh từng cặp file

**Ý tưởng:** trước hết "làm phẳng" dữ liệu thành list các cặp `(đường_dẫn, nội_dung)`. Sau đó với mỗi file chưa được xếp nhóm, duyệt **mọi file còn lại** để gom các file cùng nội dung vào cùng nhóm; nhóm có ≥ 2 file thì thêm vào đáp án.

Đây là cách trực tiếp theo định nghĩa "hai file trùng khi cùng nội dung". Ta cố ý dùng `partition` (khác `find` ở Cách 2) để baseline này làm **oracle** thứ hai khi đối chiếu.

In [8]:
def parse_flat(paths):
    """Làm phẳng: [(đường_dẫn_đầy_đủ, nội_dung), ...] (dùng partition)."""
    files = []
    for p in paths:
        parts = p.split()
        directory = parts[0]
        for token in parts[1:]:
            name, _, rest = token.partition("(")
            files.append((directory + "/" + name, rest[:-1]))
    return files


def find_dup_bruteforce(paths):
    """Cách 1: so sánh từng cặp. Thời gian O(F^2 * L) với F = số file, L = độ dài nội dung."""
    files = parse_flat(paths)
    used = [False] * len(files)
    ans = []
    for i in range(len(files)):
        if used[i]:
            continue
        group = [files[i][0]]
        used[i] = True
        for j in range(i + 1, len(files)):
            if not used[j] and files[j][1] == files[i][1]:
                group.append(files[j][0])
                used[j] = True
        if len(group) > 1:
            ans.append(group)
    return ans


print(parse_flat(ex1))
print(norm(find_dup_bruteforce(ex1)) == norm(out1), norm(find_dup_bruteforce(ex2)) == norm(out2))
assert norm(find_dup_bruteforce(ex1)) == norm(out1) and norm(find_dup_bruteforce(ex2)) == norm(out2)

[('root/a/1.txt', 'abcd'), ('root/a/2.txt', 'efgh'), ('root/c/3.txt', 'abcd'), ('root/c/d/4.txt', 'efgh'), ('root/4.txt', 'efgh')]
True True


**Độ phức tạp:** gọi `F` là tổng số file. Hai vòng lồng nhau ⇒ `O(F²)` lần so sánh nội dung; mỗi lần so sánh hai chuỗi tốn tới `O(L)` (độ dài nội dung). Tổng **`O(F² · L)`**; bộ nhớ `O(F · L)` cho danh sách file.

**Nút thắt:** với `paths.length` tới `2·10^4` và tổng độ dài `5·10^5`, số file có thể lên cỡ `10^5`; `F² = 10^10` phép so sánh — quá chậm. Ta đang **tìm lại** bạn trùng nội dung bằng cách hỏi từng file một. Nếu có cách **tra thẳng** "những file có nội dung X là ai" thì khỏi so sánh từng cặp.

## 6. Ý tưởng thuật toán / bước đột phá

Thay vì so từng cặp, dùng **bảng băm (dictionary)**: *khoá = nội dung*, *giá trị = list các đường dẫn có nội dung đó*.

1. Duyệt từng chuỗi `p` trong `paths`; tách đường dẫn thư mục và các token file.
2. Với mỗi token `tên(nội_dung)`: tính `đường_dẫn_đầy_đủ`, rồi `d[nội_dung].append(đường_dẫn_đầy_đủ)`.
3. Cuối cùng, trả mọi list trong `d.values()` có **độ dài ≥ 2**.

Mỗi file được xử lý **đúng một lần** với phép tra dictionary trung bình `O(1)` (cộng chi phí băm chuỗi nội dung `O(L)`), nên tổng thời gian **tuyến tính theo tổng độ dài input** — không còn `F²`.

Vì dictionary giữ **mọi nội dung đã gặp** trong bộ nhớ nên phải lưu nội dung làm khoá (chi phí `O(tổng độ dài nội dung)`). Phần follow-up (file khổng lồ) cho thấy khi không thể giữ nội dung ta thay khoá bằng **dấu vân tay (hash)** và lọc nhiều tầng (Cách 4).

## 7. Các cách tiếp theo

Tổng cộng 4 cách:

- Cách 1: so sánh từng cặp (mục 5).
- Cách 2: bảng băm nội dung → danh sách đường dẫn (**chính thức**).
- Cách 3: sắp xếp theo nội dung rồi gom nhóm liên tiếp (`sort` + `groupby`).
- Cách 4: lọc nhiều tầng (kích thước → đầu file → băm từng đoạn → so từng byte) cho file lớn.

---

### Cách 2 — Bảng băm: nội dung → danh sách đường dẫn

Giải thích từng đoạn:

1. `d = defaultdict(list)`.
2. `ps = p.split()`: `ps[0]` là thư mục, `ps[1:]` là các token file.
3. Với mỗi token: `i = token.find("(")`; `name = token[:i]`; `content = token[i+1:-1]` (bỏ `(` và `)` ở cuối).
4. `d[content].append(ps[0] + "/" + name)`.
5. Trả `[v for v in d.values() if len(v) > 1]`.

In [9]:
def find_dup_hash(paths):
    """Cách 2: dictionary nội dung -> list đường dẫn."""
    d = defaultdict(list)
    for p in paths:
        ps = p.split()
        for token in ps[1:]:
            i = token.find("(")
            name, content = token[:i], token[i + 1:-1]
            d[content].append(ps[0] + "/" + name)
    return [v for v in d.values() if len(v) > 1]


print(find_dup_hash(ex1))
assert norm(find_dup_hash(ex1)) == norm(out1) and norm(find_dup_hash(ex2)) == norm(out2)

[['root/a/1.txt', 'root/c/3.txt'], ['root/a/2.txt', 'root/c/d/4.txt', 'root/4.txt']]


**Code nộp LeetCode cho Cách 2:**

```python
from collections import defaultdict

class Solution:
    def findDuplicate(self, paths: List[str]) -> List[List[str]]:
        d = defaultdict(list)
        for p in paths:
            ps = p.split()
            for f in ps[1:]:
                i = f.find('(')
                name, content = f[:i], f[i + 1:-1]
                d[content].append(ps[0] + '/' + name)
        return [v for v in d.values() if len(v) > 1]
```

**Độ phức tạp:** thời gian **`O(S)`** với `S` = tổng độ dài các chuỗi trong `paths` (mỗi ký tự bị xử lý một số lần hằng); bộ nhớ `O(S)` cho dictionary và các đường dẫn.

---

### Cách 3 — Sắp xếp theo nội dung rồi gom nhóm liên tiếp

Giải thích từng đoạn:

1. Dùng `parse_flat` để có list `(nội_dung, đường_dẫn)` (đổi thứ tự cặp để sort theo nội dung).
2. `files.sort()` (hoặc `sort(key=...)`): các file **cùng nội dung đứng cạnh nhau**.
3. `groupby(files, key=lambda x: x[0])` gom các phần tử liền kề cùng nội dung.
4. Với mỗi nhóm lấy list đường dẫn; giữ nếu có ≥ 2 phần tử.

Không dùng dictionary — ý tưởng khác: "xếp hàng theo nội dung, hàng xóm giống nhau là nhóm".

In [10]:
def find_dup_sort(paths):
    """Cách 3: sort theo nội dung rồi groupby."""
    items = [(content, path) for path, content in parse_flat(paths)]
    items.sort(key=lambda x: x[0])                       # cùng nội dung -> đứng cạnh nhau
    ans = []
    for _, grp in groupby(items, key=lambda x: x[0]):
        group = [path for _, path in grp]
        if len(group) > 1:
            ans.append(group)
    return ans


print(find_dup_sort(ex1))
assert norm(find_dup_sort(ex1)) == norm(out1) and norm(find_dup_sort(ex2)) == norm(out2)

[['root/a/1.txt', 'root/c/3.txt'], ['root/a/2.txt', 'root/c/d/4.txt', 'root/4.txt']]


**Code nộp LeetCode cho Cách 3:**

```python
from itertools import groupby

class Solution:
    def findDuplicate(self, paths: List[str]) -> List[List[str]]:
        items = []
        for p in paths:
            ps = p.split()
            for f in ps[1:]:
                i = f.find('(')
                items.append((f[i + 1:-1], ps[0] + '/' + f[:i]))
        items.sort(key=lambda x: x[0])
        ans = []
        for _, grp in groupby(items, key=lambda x: x[0]):
            group = [path for _, path in grp]
            if len(group) > 1:
                ans.append(group)
        return ans
```

**Độ phức tạp:** sắp xếp `O(F log F · L)` (mỗi so sánh chuỗi tối đa `O(L)`); bộ nhớ `O(S)`. Chậm hơn Cách 2 một chút (có `log`) nhưng **không cần bảng băm** và cho thứ tự đầu ra ổn định.

---

### Cách 4 — Lọc nhiều tầng cho file lớn (follow-up)

Giả sử mỗi file nặng cả GB, không thể đọc hết rồi lưu làm khoá. Phương pháp thực tế:

1. **Nhóm theo kích thước** (rẻ: chỉ đọc metadata). File khác kích thước chắc chắn khác nội dung. Bỏ nhóm size 1.
2. Trong mỗi nhóm cùng kích thước, **nhóm theo "đầu file"** (đọc chunk đầu, ví dụ 1 KB). Bỏ nhóm 1.
3. Còn lại **nhóm theo băm toàn bộ file**, băm **từng đoạn** bằng `update()` (đọc 1 KB mỗi lần, RAM `O(1KB)`).
4. **Kiểm tra lại từng byte** trong mỗi nhóm cùng băm để loại trường hợp **đụng hàm băm (false positive)**.

Ở notebook ta mô phỏng "file" bằng chuỗi nội dung trong RAM và "đọc 1 KB" bằng đoạn `chunk` ký tự. Hàm băm được truyền vào như tham số để ở mục 13 ta thay bằng hàm băm **yếu** và thấy bước kiểm tra byte cứu ta.

In [11]:
def sha_chunked(content, chunk=4):
    """Băm sha256 từng đoạn `chunk` ký tự (mô phỏng đọc file theo từng khối)."""
    h = hashlib.sha256()
    for k in range(0, len(content), chunk):
        h.update(content[k:k + chunk].encode())
    return h.hexdigest()


def find_dup_staged(paths, chunk=4, hash_fn=sha_chunked):
    """Cách 4: size -> đầu file -> băm -> so từng byte."""
    by_size = defaultdict(list)
    for path, content in parse_flat(paths):
        by_size[len(content)].append((path, content))
    ans = []
    for lst in by_size.values():
        if len(lst) < 2:
            continue                                         # tầng 1: khác kích thước
        by_head = defaultdict(list)
        for path, content in lst:
            by_head[content[:chunk]].append((path, content))
        for lst2 in by_head.values():
            if len(lst2) < 2:
                continue                                     # tầng 2: khác đầu file
            by_hash = defaultdict(list)
            for path, content in lst2:
                by_hash[hash_fn(content, chunk)].append((path, content))
            for lst3 in by_hash.values():
                if len(lst3) < 2:
                    continue                                 # tầng 3: khác băm
                buckets = []                                 # tầng 4: so từng byte với đại diện
                for path, content in lst3:
                    for rep, plist in buckets:
                        if rep == content:
                            plist.append(path)
                            break
                    else:
                        buckets.append((content, [path]))
                ans.extend(plist for _, plist in buckets if len(plist) > 1)
    return ans


print(find_dup_staged(ex1))
assert norm(find_dup_staged(ex1)) == norm(out1) and norm(find_dup_staged(ex2)) == norm(out2)

[['root/a/1.txt', 'root/c/3.txt'], ['root/a/2.txt', 'root/c/d/4.txt', 'root/4.txt']]


**Độ phức tạp (mô hình RAM):** thời gian `O(S)` kỳ vọng (mỗi tầng chỉ xử lý phần còn "nghi ngờ"); bộ nhớ chỉ giữ **băm/đầu file** thay vì nội dung đầy đủ khi dùng file thật. Tốn thêm 2 lần đọc đĩa với những file thật sự trùng (băm rồi so byte).

**Câu hỏi follow-up và gợi ý trả lời:**

- *DFS hay BFS để duyệt hệ thống file thật?* Cả hai đều đúng (mỗi thư mục thăm một lần). DFS dùng ít bộ nhớ hơn khi cây rộng (stack theo độ sâu); BFS dùng hàng đợi theo độ rộng. Thường chọn DFS (đệ quy hoặc stack).
- *File GB?* Không đọc nguyên file: lọc theo size, đầu file, băm theo khối (Cách 4).
- *Chỉ đọc 1 KB mỗi lần?* Băm tăng dần bằng `update` trên từng khối 1 KB.
- *Chi phí lớn nhất?* Đọc đĩa (I/O) khi băm/so byte file lớn; bộ nhớ: bảng băm các `(size, hash) → danh sách đường dẫn`. Tối ưu: lọc nhiều tầng, đọc song song, băm trên đầu/giữa/cuối trước.
- *Tránh false positive?* So sánh **từng byte** các file cùng băm (tầng 4).

## 8. TRACE từng bước

### 8.1. Trace Cách 2 với ví dụ 1

`paths = ["root/a 1.txt(abcd) 2.txt(efgh)", "root/c 3.txt(abcd)", "root/c/d 4.txt(efgh)", "root 4.txt(efgh)"]`

| chuỗi | thư mục | token | name | content | `d` sau bước |
|---|---|---|---|---|---|
| 1 | `root/a` | `1.txt(abcd)` | `1.txt` | `abcd` | `{abcd: [root/a/1.txt]}` |
| 1 | `root/a` | `2.txt(efgh)` | `2.txt` | `efgh` | `+ {efgh: [root/a/2.txt]}` |
| 2 | `root/c` | `3.txt(abcd)` | `3.txt` | `abcd` | `abcd: [root/a/1.txt, root/c/3.txt]` |
| 3 | `root/c/d` | `4.txt(efgh)` | `4.txt` | `efgh` | `efgh: [root/a/2.txt, root/c/d/4.txt]` |
| 4 | `root` | `4.txt(efgh)` | `4.txt` | `efgh` | `efgh: [..., root/4.txt]` |

Cuối cùng cả hai list đều có ≥ 2 phần tử ⇒ 2 nhóm. Hàm `trace_hash` bên dưới in từng bước.

In [12]:
def trace_hash(paths):
    d = defaultdict(list)
    for p in paths:
        ps = p.split()
        print(f"chuỗi {p!r}\n  thư mục = {ps[0]!r}, các token = {ps[1:]}")
        for token in ps[1:]:
            i = token.find("(")
            name, content = token[:i], token[i + 1:-1]
            d[content].append(ps[0] + "/" + name)
            print(f"  token {token!r}: find('(')={i} name={name!r} content={content!r} -> d[{content!r}] = {d[content]}")
    result = [v for v in d.values() if len(v) > 1]
    print("các nhóm có >= 2 phần tử:", result)
    return result


r = trace_hash(ex1)
assert norm(r) == norm(out1)

chuỗi 'root/a 1.txt(abcd) 2.txt(efgh)'
  thư mục = 'root/a', các token = ['1.txt(abcd)', '2.txt(efgh)']
  token '1.txt(abcd)': find('(')=5 name='1.txt' content='abcd' -> d['abcd'] = ['root/a/1.txt']
  token '2.txt(efgh)': find('(')=5 name='2.txt' content='efgh' -> d['efgh'] = ['root/a/2.txt']
chuỗi 'root/c 3.txt(abcd)'
  thư mục = 'root/c', các token = ['3.txt(abcd)']
  token '3.txt(abcd)': find('(')=5 name='3.txt' content='abcd' -> d['abcd'] = ['root/a/1.txt', 'root/c/3.txt']
chuỗi 'root/c/d 4.txt(efgh)'
  thư mục = 'root/c/d', các token = ['4.txt(efgh)']
  token '4.txt(efgh)': find('(')=5 name='4.txt' content='efgh' -> d['efgh'] = ['root/a/2.txt', 'root/c/d/4.txt']
chuỗi 'root 4.txt(efgh)'
  thư mục = 'root', các token = ['4.txt(efgh)']
  token '4.txt(efgh)': find('(')=5 name='4.txt' content='efgh' -> d['efgh'] = ['root/a/2.txt', 'root/c/d/4.txt', 'root/4.txt']
các nhóm có >= 2 phần tử: [['root/a/1.txt', 'root/c/3.txt'], ['root/a/2.txt', 'root/c/d/4.txt', 'root/4.txt']]


### 8.2. Trace Cách 3 (sort + groupby)

Với ví dụ 1, sau khi sắp xếp theo nội dung: `abcd` đứng trước `efgh` (so sánh từ điển). Danh sách thành:

```
(abcd, root/a/1.txt) (abcd, root/c/3.txt) | (efgh, root/a/2.txt) (efgh, root/c/d/4.txt) (efgh, root/4.txt)
```

`groupby` cắt thành hai nhóm tại chỗ nội dung đổi từ `abcd` sang `efgh`.

In [13]:
def trace_sort(paths):
    items = [(content, path) for path, content in parse_flat(paths)]
    items.sort(key=lambda x: x[0])
    print("sau khi sort:", items)
    ans = []
    for key, grp in groupby(items, key=lambda x: x[0]):
        group = [path for _, path in grp]
        print(f"  nhóm nội dung {key!r}: {group} -> {'GIỮ' if len(group) > 1 else 'bỏ (chỉ 1 file)'}")
        if len(group) > 1:
            ans.append(group)
    return ans


assert norm(trace_sort(ex1)) == norm(out1)
print("-----")
trace_sort(["root a.txt(x) b.txt(y)", "root/z c.txt(x)"])

sau khi sort: [('abcd', 'root/a/1.txt'), ('abcd', 'root/c/3.txt'), ('efgh', 'root/a/2.txt'), ('efgh', 'root/c/d/4.txt'), ('efgh', 'root/4.txt')]
  nhóm nội dung 'abcd': ['root/a/1.txt', 'root/c/3.txt'] -> GIỮ
  nhóm nội dung 'efgh': ['root/a/2.txt', 'root/c/d/4.txt', 'root/4.txt'] -> GIỮ
-----
sau khi sort: [('x', 'root/a.txt'), ('x', 'root/z/c.txt'), ('y', 'root/b.txt')]
  nhóm nội dung 'x': ['root/a.txt', 'root/z/c.txt'] -> GIỮ
  nhóm nội dung 'y': ['root/b.txt'] -> bỏ (chỉ 1 file)


[['root/a.txt', 'root/z/c.txt']]

## 9. Code nộp LeetCode

Cách 2 (chính thức): đúng `class Solution`, phương thức `findDuplicate(self, paths: List[str]) -> List[List[str]]`.

In [14]:
class Solution:
    def findDuplicate(self, paths: List[str]) -> List[List[str]]:
        d = defaultdict(list)
        for p in paths:
            ps = p.split()
            for f in ps[1:]:
                i = f.find('(')
                name, content = f[:i], f[i + 1:-1]
                d[content].append(ps[0] + '/' + name)
        return [v for v in d.values() if len(v) > 1]


sol = Solution()
print(sol.findDuplicate(ex1))
print(sol.findDuplicate(ex2))
assert norm(sol.findDuplicate(ex1)) == norm(out1) and norm(sol.findDuplicate(ex2)) == norm(out2)

[['root/a/1.txt', 'root/c/3.txt'], ['root/a/2.txt', 'root/c/d/4.txt', 'root/4.txt']]
[['root/a/1.txt', 'root/c/3.txt'], ['root/a/2.txt', 'root/c/d/4.txt']]


## 10. Vì sao đúng

- **Phân tích chuỗi đúng:** mỗi `paths[i]` gồm thư mục (không chứa dấu cách) rồi các token `tên(nội_dung)` ngăn cách bằng đúng một dấu cách; tên file và nội dung đều **không chứa dấu cách** (đề chỉ cho dấu cách làm ngăn cách) và nội dung không chứa `(`/`)` như một phần dữ liệu nên `find("(")` tìm đúng ranh giới `tên | nội dung` và `[i+1:-1]` cắt đúng nội dung (bỏ `(` đầu và `)` cuối).
- **Khoá nhóm đúng định nghĩa:** hai file trùng ⇔ cùng nội dung ⇔ rơi vào cùng khoá `d[content]`. Do đó mỗi list trong `d` chính là một **lớp tương đương** theo quan hệ "cùng nội dung".
- **Đủ và không thừa:** mỗi file được thêm vào **đúng một** list (list của nội dung nó) ⇒ đường dẫn xuất hiện đúng một lần trong tập hợp các list. Giữ list có ≥ 2 phần tử ⇒ mỗi nhóm đáp án có ít nhất hai file, và mọi cặp file trùng đều nằm trong một nhóm.
- **Đường dẫn đầy đủ:** `thư_mục + "/" + tên` đúng định dạng đề (kể cả `m = 0`, thư mục là `root`).
- **Thứ tự bất kỳ:** đề cho phép, nên dictionary (thứ tự chèn) hay sort đều hợp lệ.
- **Các cách khác:** Cách 1 duyệt định nghĩa trực tiếp; Cách 3 đúng vì sau sort các nội dung bằng nhau liền kề; Cách 4 chỉ **lọc thêm điều kiện cần** (cùng size, cùng đầu, cùng băm) rồi **xác nhận bằng so byte** nên không có false positive/negative.

## 11. Phân tích Time / Space Complexity

Gọi `S` = tổng độ dài các chuỗi (≤ `5·10^5`), `F` = số file, `L` = độ dài nội dung lớn nhất.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. So từng cặp | `O(F² · L)` | `O(S)` | chậm, dùng làm oracle |
| 2. Dictionary nội dung | `O(S)` | `O(S)` | **chính thức** |
| 3. Sort + groupby | `O(F log F · L)` | `O(S)` | không cần dict |
| 4. Lọc nhiều tầng | `O(S)` kỳ vọng | `O(F)` nếu chỉ giữ băm | dành cho file lớn |

Vì mỗi ký tự của input được đọc một số lần hằng trong Cách 2 nên `O(S)` là tối ưu theo bậc (bắt buộc phải đọc input).

## 12. Bộ test

Chiến lược (bài **nhiều đáp án đúng**: chuẩn hoá + hàm kiểm tra hợp lệ):

1. **Ví dụ đề** (so sau chuẩn hoá `norm`).
2. **Edge case:** không có nhóm trùng (`[]`), chỉ một chuỗi mà hai file cùng nội dung, thư mục `root` (`m = 0`), một nhóm rất lớn, tên file có nhiều dấu chấm.
3. **Hàm kiểm tra hợp lệ** `is_valid_answer` độc lập (dùng `regex` để phân tích, tự dựng nhóm đúng bằng so sánh cặp): kiểm tra mỗi nhóm ≥ 2 file cùng nội dung, mỗi đường dẫn xuất hiện đúng một lần, không sót file trùng.
4. **Đối chiếu nhiều input**: sinh ngẫu nhiên có seed đúng ràng buộc; so Cách 1, 2, 3, 4 và `Solution` bằng `norm` + `is_valid_answer`.
5. **Hiệu năng:** input lớn (12000 chuỗi, tổng độ dài gần `5·10^5`).

In [15]:
TOKEN_RE = re.compile(r"(\S+?)\((\S*)\)$")


def file_table(paths):
    """Oracle phân tích bằng regex: {đường_dẫn_đầy_đủ: nội_dung}. Kiểm tra luôn ràng buộc không trùng tên."""
    table = {}
    for p in paths:
        directory, *tokens = p.split(" ")
        for t in tokens:
            m = TOKEN_RE.match(t)
            assert m, ("token sai định dạng", t)
            full = directory + "/" + m.group(1)
            assert full not in table, ("đường dẫn lặp", full)
            table[full] = m.group(2)
    return table


def is_valid_answer(paths, ans):
    """Hàm kiểm tra tính hợp lệ, không phụ thuộc cách giải."""
    table = file_table(paths)
    seen = set()
    for g in ans:
        assert len(g) >= 2, ("nhóm quá nhỏ", g)
        for path in g:
            assert path in table, ("đường dẫn không tồn tại", path)
            assert path not in seen, ("đường dẫn xuất hiện hai lần", path)
            seen.add(path)
        assert len({table[path] for path in g}) == 1, ("nhóm lẫn nội dung", g)
    # không sót: mọi file mà nội dung xuất hiện >= 2 lần phải nằm trong đáp án (so từng cặp, không dùng dict-group)
    paths_list = list(table)
    for i, a in enumerate(paths_list):
        has_twin = any(table[a] == table[b] for j, b in enumerate(paths_list) if j != i)
        assert (a in seen) == has_twin, ("sót/thừa file", a)
    return True


# Khối "ví dụ đề + edge case"
assert is_valid_answer(ex1, out1) and is_valid_answer(ex2, out2)
IMPLS = [find_dup_bruteforce, find_dup_hash, find_dup_sort, find_dup_staged, lambda ps: Solution().findDuplicate(ps)]
NAMES = ["bruteforce", "hash", "sort", "staged", "Solution"]

cases = [
    (ex1, out1),
    (ex2, out2),
    (["root a.txt(x) b.txt(y)"], []),                                   # không trùng
    (["root a.txt(x) b.txt(x)"], [["root/a.txt", "root/b.txt"]]),       # cùng thư mục, root (m = 0)
    (["root/a f.txt(zz)", "root/b f.txt(zz)", "root/c f.txt(zz)", "root/d f.txt(zz)"],
     [["root/a/f.txt", "root/b/f.txt", "root/c/f.txt", "root/d/f.txt"]]),   # nhóm 4 file
    (["root/a x.y.txt(k1) x.z.txt(k2)", "root/b q.txt(k1)"], [["root/a/x.y.txt", "root/b/q.txt"]]),   # tên có nhiều dấu chấm
    (["root a(1) b(2) c(1) d(2) e(3)"], [["root/a", "root/c"], ["root/b", "root/d"]]),
]
for paths, expected in cases:
    for f, name in zip(IMPLS, NAMES):
        got = f(paths)
        assert norm(got) == norm(expected), (name, paths, got)
        assert is_valid_answer(paths, got)
print(len(cases), "ca cố định: cả 5 cách đều đúng (so sau chuẩn hoá + hợp lệ)")

7 ca cố định: cả 5 cách đều đúng (so sau chuẩn hoá + hợp lệ)


In [16]:
def gen_paths(rng, n_dirs, max_files, pool):
    """Sinh input hợp lệ: thư mục khác nhau, tên file khác nhau trong cùng thư mục, nội dung lấy từ `pool`."""
    dirs = set()
    parts = ["a", "b", "c", "d"]
    while len(dirs) < n_dirs:
        dirs.add("root" + "".join("/" + rng.choice(parts) for _ in range(rng.randint(0, 3))))
    out = []
    for d in dirs:
        k = rng.randint(1, max_files)
        names = [f"{rng.choice('xyz')}{i}.txt" for i in range(1, k + 1)]      # chỉ số i đảm bảo khác nhau
        out.append(d + " " + " ".join(f"{nm}({rng.choice(pool)})" for nm in names))
    rng.shuffle(out)
    return out


rng = random.Random(609)
count = 0
for _ in range(600):
    pool = ["".join(rng.choice("ab12") for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 8))]
    paths = gen_paths(rng, rng.randint(1, 8), rng.randint(1, 4), pool)
    base = norm(find_dup_bruteforce(paths))
    for f, name in zip(IMPLS, NAMES):
        got = f(paths)
        assert norm(got) == base, (name, paths)
        assert is_valid_answer(paths, got), (name, paths)
    count += 1
print(count, "input ngẫu nhiên có seed: 5 cách khớp nhau và đều hợp lệ")

600 input ngẫu nhiên có seed: 5 cách khớp nhau và đều hợp lệ


In [17]:
# Input LỚN gần cận ràng buộc: 12000 chuỗi, tổng độ dài < 5*10^5, nhiều nội dung trùng (không dùng Cách 1: F^2 quá chậm)
rng = random.Random(1234)
pool = ["".join(rng.choice("abcdefghij0123456789") for _ in range(rng.randint(3, 12))) for _ in range(3000)]
big = []
for i in range(12000):
    depth = rng.randint(0, 3)
    d = "root" + "".join("/" + rng.choice(["a", "b", "c", "d", "e", "f"]) for _ in range(depth)) + f"/u{i}"
    k = rng.randint(1, 2)
    big.append(d + " " + " ".join(f"f{j}.txt({rng.choice(pool)})" for j in range(k)))
print("số chuỗi:", len(big), "| tổng độ dài:", sum(map(len, big)), "| max độ dài:", max(map(len, big)))

results = {}
for f, name in zip(IMPLS[1:], NAMES[1:]):
    t0 = time.perf_counter()
    res = f(big)
    results[name] = norm(res)
    print(f"{name:<9} {time.perf_counter() - t0:.3f}s  số nhóm: {len(res)}")
assert results["hash"] == results["sort"] == results["staged"] == results["Solution"]
assert is_valid_answer(big[:3000], Solution().findDuplicate(big[:3000]))      # kiểm tra hợp lệ trên một phần (oracle O(F^2))
print("input lớn: các cách khớp nhau; hợp lệ trên phần đầu")

số chuỗi: 12000 | tổng độ dài: 452229 | max độ dài: 59
hash      0.016s  số nhóm: 2938
sort      0.023s  số nhóm: 2938
staged    0.037s  số nhóm: 2938
Solution  0.012s  số nhóm: 2938


input lớn: các cách khớp nhau; hợp lệ trên phần đầu


## 13. Các lỗi người mới hay gặp

### Lỗi 1: quên lọc nhóm chỉ có 1 file

Trả về `list(d.values())` thay vì chỉ các nhóm `len > 1`: mọi file **không trùng** cũng bị đưa vào đáp án dưới dạng nhóm một phần tử.

In [18]:
def wrong_no_filter(paths):
    d = defaultdict(list)
    for p in paths:
        ps = p.split()
        for f in ps[1:]:
            i = f.find("(")
            d[f[i + 1:-1]].append(ps[0] + "/" + f[:i])
    return list(d.values())                    # SAI: thiếu `if len(v) > 1`


bad = wrong_no_filter(["root a.txt(x) b.txt(y)", "root/z c.txt(x)"])
print(bad, "(đúng phải chỉ có [root/a.txt, root/z/c.txt])")
try:
    is_valid_answer(["root a.txt(x) b.txt(y)", "root/z c.txt(x)"], bad)
except AssertionError as e:
    print("hàm kiểm tra bắt được:", e)

[['root/a.txt', 'root/z/c.txt'], ['root/b.txt']] (đúng phải chỉ có [root/a.txt, root/z/c.txt])
hàm kiểm tra bắt được: ('nhóm quá nhỏ', ['root/b.txt'])


**Sửa:** `[v for v in d.values() if len(v) > 1]`.

### Lỗi 2: quên ghép thư mục vào tên file

Đề yêu cầu **đường dẫn đầy đủ** `thư_mục/tên`. Nếu chỉ trả `name` (ví dụ `1.txt`), hai file khác thư mục nhưng cùng tên sẽ bị lẫn và đáp án sai định dạng.

In [19]:
def wrong_no_dir(paths):
    d = defaultdict(list)
    for p in paths:
        ps = p.split()
        for f in ps[1:]:
            i = f.find("(")
            d[f[i + 1:-1]].append(f[:i])       # SAI: thiếu ps[0] + "/"
    return [v for v in d.values() if len(v) > 1]


bad = wrong_no_dir(ex1)
print(bad, "-> mất thông tin thư mục")
print(norm(bad) == norm(out1))

[['1.txt', '3.txt'], ['2.txt', '4.txt', '4.txt']] -> mất thông tin thư mục
False


**Sửa:** `ps[0] + "/" + name`.

### Lỗi 3: dùng chung **một** list cho mọi khoá (`dict.fromkeys(keys, [])`)

Khi muốn khởi tạo trước các khoá, viết `d = dict.fromkeys(contents, [])` tạo **một list duy nhất** mà mọi khoá cùng trỏ tới ⇒ thêm vào khoá này là thêm vào tất cả. Đây là lỗi "alias" kinh điển của Python.

In [20]:
d = dict.fromkeys(["abcd", "efgh"], [])        # SAI: cả hai khoá dùng CHUNG một list
d["abcd"].append("root/a/1.txt")
print(d, "->", d["abcd"] is d["efgh"])

d = {k: [] for k in ["abcd", "efgh"]}          # ĐÚNG: mỗi khoá một list riêng
d["abcd"].append("root/a/1.txt")
print(d, "->", d["abcd"] is d["efgh"])

{'abcd': ['root/a/1.txt'], 'efgh': ['root/a/1.txt']} -> True
{'abcd': ['root/a/1.txt'], 'efgh': []} -> False


**Sửa:** dùng `defaultdict(list)` hoặc `{k: [] for k in keys}`.

### Lỗi 4: tách sai ranh giới nội dung (`[i:-1]` thay vì `[i+1:-1]`)

Nếu cắt `token[i:-1]` thì nội dung giữ lại dấu `(` ở đầu. Vì lỗi **áp dụng đồng đều** nên việc gom nhóm vẫn đúng, nhưng nếu bạn dùng nội dung ở bước khác (so sánh, tính kích thước, in ra) thì dữ liệu lệch. Ví dụ kích thước nội dung bị thừa 1.

In [21]:
token = "1.txt(abcd)"
i = token.find("(")
print(repr(token[i:-1]), len(token[i:-1]), "(SAI: thừa dấu '(')")
print(repr(token[i + 1:-1]), len(token[i + 1:-1]), "(ĐÚNG)")

'(abcd' 5 (SAI: thừa dấu '(')
'abcd' 4 (ĐÚNG)


### Lỗi 5: test bằng so sánh cứng thứ tự

`assert Solution().findDuplicate(ex1) == out1` có thể **đúng hôm nay rồi sai** khi đổi cách cài đặt (ví dụ đổi từ dictionary sang sort) dù đáp án vẫn hợp lệ. Hãy dùng `norm` (chuẩn hoá) hoặc hàm kiểm tra tính hợp lệ như mục 12.

In [22]:
mine = find_dup_sort(ex1)
print(mine)
print("so cứng:", mine == out1, "| sau chuẩn hoá:", norm(mine) == norm(out1), "| hợp lệ:", is_valid_answer(ex1, mine))

[['root/a/1.txt', 'root/c/3.txt'], ['root/a/2.txt', 'root/c/d/4.txt', 'root/4.txt']]
so cứng: False | sau chuẩn hoá: True | hợp lệ: True


## 14. Mẹo & nhận dạng pattern

- **"Tìm các phần tử trùng/giống nhau rồi gom nhóm"** ⇒ **group by bằng dictionary** với khoá là "dấu hiệu nhận biết": nội dung (bài này), chuỗi đã sort (bài 49 Group Anagrams), tổng kiểm (checksum)...
- **Mẫu `defaultdict(list)`** + `d[key].append(item)` rồi lọc `len(v) > 1`.
- **Phân tích chuỗi có cấu trúc:** `split()` ra các phần, `find`/`partition` để cắt tại ký tự phân cách, slicing để bỏ ký tự thừa; luôn tự kiểm tra bằng một token mẫu.
- **Bài nhiều đáp án đúng:** chuẩn hoá (`sorted`) hoặc viết hàm kiểm tra tính hợp lệ.
- **Follow-up về dữ liệu lớn:** nghĩ "lọc rẻ trước, đắt sau": kích thước → đầu file → băm → so byte.
- **`groupby` chỉ gom phần tử liền kề:** nhớ `sort` theo cùng khoá trước.
- Bài họ hàng: 49 (Group Anagrams), 1048, 1948 (Delete Duplicate Folders in System), 38.

## 15. Biến thể

1. **Chỉ giữ nhóm trải rộng qua ≥ 2 thư mục khác nhau** (bỏ các nhóm mà mọi bản sao nằm trong cùng một thư mục).
2. **Tính dung lượng có thể tiết kiệm** nếu xoá các bản sao (giữ lại một bản mỗi nhóm): `Σ (số_bản − 1) · kích_thước`.
3. **Duplicate theo tên** thay vì nội dung (khoá = tên file) — cùng khung group by.
4. **Thư mục trùng (bài 1948, Hard):** hai thư mục trùng nếu cấu trúc con y hệt — cần "tuần tự hoá" cây thành chuỗi chuẩn rồi group by.
5. **Streaming/lớn hơn RAM:** Cách 4 (lọc nhiều tầng) hoặc băm rồi ghi ra đĩa theo bucket.

Minh hoạ nhanh biến thể 1 và 2:

In [23]:
def across_dirs(paths):
    """Biến thể 1: chỉ giữ nhóm có file ở >= 2 thư mục khác nhau."""
    return [g for g in find_dup_hash(paths) if len({p.rsplit("/", 1)[0] for p in g}) >= 2]


def bytes_saved(paths):
    """Biến thể 2: tổng ký tự tiết kiệm được nếu giữ 1 bản mỗi nhóm."""
    sizes = {path: len(content) for path, content in parse_flat(paths)}
    return sum((len(g) - 1) * sizes[g[0]] for g in find_dup_hash(paths))


p = ["root/a x.txt(hello) y.txt(hello)", "root/b z.txt(world)", "root/c w.txt(world)"]
print(find_dup_hash(p))                 # hello trùng trong cùng thư mục; world trùng khác thư mục
print(across_dirs(p))                   # chỉ còn nhóm world
print(bytes_saved(p))                   # (2-1)*5 + (2-1)*5 = 10
assert norm(across_dirs(p)) == [["root/b/z.txt", "root/c/w.txt"]] and bytes_saved(p) == 10

[['root/a/x.txt', 'root/a/y.txt'], ['root/b/z.txt', 'root/c/w.txt']]
[['root/b/z.txt', 'root/c/w.txt']]
10


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. So từng cặp | duyệt mọi cặp file, so nội dung | `O(F² · L)` | `O(S)` | chỉ để học / oracle test |
| 2. Dictionary | khoá = nội dung, giá trị = list đường dẫn | `O(S)` | `O(S)` | **mặc định khi nộp bài** |
| 3. Sort + groupby | sort theo nội dung, gom liền kề | `O(F log F · L)` | `O(S)` | không muốn dùng dict, cần thứ tự ổn định |
| 4. Lọc nhiều tầng | size → đầu → băm từng khối → so byte | `O(S)` kỳ vọng | `O(F)` (chỉ giữ băm) | file rất lớn, đọc theo khối, cần chống false positive |

## 17. Bài tập vận dụng (dễ → khó)

Mỗi bài có cell TODO để bạn tự làm (cell **chưa** được gọi, chỉ gán `None` hoặc `def` rỗng). Đáp án ở mục 19, gợi ý ở mục 18.

### Bài 1 (đoán output)

Với `token = "log.v2.txt(a1b2)"`, `i = token.find("(")`, dự đoán giá trị của: (1) `i`; (2) `token[:i]`; (3) `token[i + 1:-1]`; (4) `"root/x" + "/" + token[:i]`. Ghi vào list `bai1_du_doan` theo đúng thứ tự.

In [24]:
# TODO Bài 1: bai1_du_doan = [i, ten, noi_dung, duong_dan]
bai1_du_doan = None

### Bài 2 (tính tay)

`paths = ["root a(1) b(2)", "root/x c(1) d(3)", "root/y e(2) f(1)"]`. (a) Có bao nhiêu **nhóm** trùng? (b) Nhóm lớn nhất có bao nhiêu file? Ghi `(a, b)` vào `bai2`.

In [25]:
# TODO Bài 2: bai2 = (so_nhom, kich_thuoc_nhom_lon_nhat)
bai2 = None

### Bài 3 (điền TODO)

Hoàn thành hàm `parse_token(directory, token)` trả `(đường_dẫn_đầy_đủ, nội_dung)`.

In [26]:
def parse_token(directory, token):
    # TODO: tìm vị trí '(' ; name = phần trước ; content = phần giữa '(' và ')' ; trả (directory + '/' + name, content)
    pass

### Bài 4 (tự viết lại)

Không nhìn mục 9, tự viết hàm `find_duplicate(paths)` theo Cách 2 (dictionary nội dung → list đường dẫn, lọc nhóm ≥ 2).

In [27]:
# TODO Bài 4: tự viết lại
def find_duplicate(paths):
    pass

### Bài 5 (biến thể / mini challenge)

Viết hàm `dup_across_dirs(paths)`: giống bài gốc nhưng **chỉ trả các nhóm có file nằm ở ít nhất hai thư mục khác nhau**. (Gợi ý: thư mục của `"root/a/1.txt"` là `"root/a"`.)

In [28]:
# TODO Bài 5
def dup_across_dirs(paths):
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `token.find("(")` đếm từ 0; `"log.v2.txt"` có 10 ký tự.
- Mức 2: `token[:i]` lấy 10 ký tự đầu; `token[i+1:-1]` bỏ `(` và `)`.
- Mức 3: `i = 10`; tên `"log.v2.txt"`; nội dung `"a1b2"`; đường dẫn `"root/x/log.v2.txt"`.

**Bài 2**
- Mức 1: liệt kê các nội dung: `1` ở `a`, `c`, `f`; `2` ở `b`, `e`; `3` ở `d`.
- Mức 2: nhóm có ≥ 2 file là nhóm trùng.
- Mức 3: nhóm `1` (3 file), nhóm `2` (2 file), nhóm `3` bị bỏ (1 file) ⇒ `(2, 3)`.

**Bài 3**
- Mức 1: `i = token.find("(")`.
- Mức 2: `name = token[:i]`, `content = token[i + 1:-1]`.
- Mức 3: `return directory + "/" + name, content`.

**Bài 4**
- Mức 1: `defaultdict(list)` và hai vòng lặp (chuỗi, token).
- Mức 2: `ps = p.split()`; `ps[0]` là thư mục.
- Mức 3: cuối hàm `return [v for v in d.values() if len(v) > 1]`.

**Bài 5**
- Mức 1: tái dùng nhóm của bài gốc rồi lọc.
- Mức 2: thư mục của một đường dẫn: `path.rsplit("/", 1)[0]`.
- Mức 3: `len({thư_mục_của(p) for p in nhóm}) >= 2`.

## 19. Đáp án tham khảo

Chạy kiểm tra từng đáp án bằng `assert`.

In [29]:
# Bài 1
token = "log.v2.txt(a1b2)"
i = token.find("(")
dap_an_1 = [i, token[:i], token[i + 1:-1], "root/x" + "/" + token[:i]]
print(dap_an_1)
assert dap_an_1 == [10, "log.v2.txt", "a1b2", "root/x/log.v2.txt"]

# Bài 2
paths2 = ["root a(1) b(2)", "root/x c(1) d(3)", "root/y e(2) f(1)"]
groups = find_dup_hash(paths2)
bai2_dap_an = (len(groups), max(map(len, groups)))
print(groups, bai2_dap_an)
assert bai2_dap_an == (2, 3) and is_valid_answer(paths2, groups)

# Bài 3
def parse_token(directory, token):
    i = token.find("(")
    return directory + "/" + token[:i], token[i + 1:-1]


assert parse_token("root/c", "3.txt(abcd)") == ("root/c/3.txt", "abcd")
assert parse_token("root", "a.b.txt(Z9)") == ("root/a.b.txt", "Z9")
print("Bài 3: OK")

[10, 'log.v2.txt', 'a1b2', 'root/x/log.v2.txt']
[['root/a', 'root/x/c', 'root/y/f'], ['root/b', 'root/y/e']] (2, 3)
Bài 3: OK


In [30]:
# Bài 4
def find_duplicate(paths):
    d = defaultdict(list)
    for p in paths:
        ps = p.split()
        for token in ps[1:]:
            path, content = parse_token(ps[0], token)
            d[content].append(path)
    return [v for v in d.values() if len(v) > 1]


assert norm(find_duplicate(ex1)) == norm(out1) and norm(find_duplicate(ex2)) == norm(out2)
rng = random.Random(4)
for _ in range(300):
    pool = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 3))) for _ in range(rng.randint(1, 6))]
    ps = gen_paths(rng, rng.randint(1, 6), 3, pool)
    assert is_valid_answer(ps, find_duplicate(ps))
print("Bài 4: OK")

# Bài 5
def dup_across_dirs(paths):
    return [g for g in find_duplicate(paths) if len({p.rsplit("/", 1)[0] for p in g}) >= 2]


p = ["root/a x.txt(hello) y.txt(hello)", "root/b z.txt(world)", "root/c w.txt(world)"]
assert norm(dup_across_dirs(p)) == [["root/b/z.txt", "root/c/w.txt"]]
assert dup_across_dirs(["root a.txt(x) b.txt(x)"]) == []                 # trùng nhưng cùng một thư mục
assert norm(dup_across_dirs(ex1)) == norm(out1)
rng = random.Random(5)
for _ in range(300):
    pool = ["".join(rng.choice("ab") for _ in range(rng.randint(1, 3))) for _ in range(rng.randint(1, 6))]
    ps = gen_paths(rng, rng.randint(1, 6), 3, pool)
    assert norm(dup_across_dirs(ps)) == norm(across_dirs(ps))
print("Bài 5: OK")

Bài 4: OK
Bài 5: OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template "group by khoá rồi lọc nhóm ≥ 2":**

```python
from collections import defaultdict

class Solution:
    def findDuplicate(self, paths: List[str]) -> List[List[str]]:
        d = defaultdict(list)                       # khoá: nội dung -> list đường dẫn
        for p in paths:
            ps = p.split()                          # ps[0] = thư mục, ps[1:] = các file
            for f in ps[1:]:
                i = f.find('(')
                name, content = f[:i], f[i + 1:-1]  # bỏ '(' đầu và ')' cuối
                d[content].append(ps[0] + '/' + name)
        return [v for v in d.values() if len(v) > 1]
```

**Checklist trước khi submit:**

- [ ] Khoá nhóm là **nội dung** (không phải tên file/thư mục)?
- [ ] Đã ghép `thư_mục + "/" + tên` thành đường dẫn đầy đủ (kể cả `root`)?
- [ ] Cắt nội dung bằng `[i + 1:-1]` (bỏ cả `(` lẫn `)`)?
- [ ] Đã **lọc** nhóm có `len > 1`?
- [ ] Không dùng `dict.fromkeys(keys, [])` (alias)? Dùng `defaultdict(list)` hoặc dict comprehension?
- [ ] Test bằng **chuẩn hoá** hoặc hàm kiểm tra hợp lệ (không so cứng thứ tự)?
- [ ] Đã nghĩ tới follow-up: file lớn → size → đầu file → băm từng khối → so byte?